## Loading environment variable.

In [ ]:
import os
os.system("ollama run llama3")


import requests 

response = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model":"llama3", 
        "messages": [ 
            {
            "role":"user", "content":"Hello"}
                    ], 
        "stream":False
    }
)
print(response.json())

successfull

## Create BigCompany Inc.'s Expense Database

In [ ]:

EXPENSES_DB = {
    "EXP-001": {
        "employee": "Alice Johnson",
        "department": "Engineering",
        "description": "Team lunch at downtown restaurant",
        "amount": 127.50,
        "category": "meals",
        "status": "pending",
    },
    "EXP-002": {
        "employee": "Bob Smith",
        "department": "Sales",
        "description": "Flight to New York for client meeting",
        "amount": 1250.00,
        "category": "travel",
        "status": "pending",
    },
    "EXP-003": {
        "employee": "Carol Davis",
        "department": "Marketing",
        "description": "Annual Adobe Creative Cloud subscription",
        "amount": 659.88,
        "category": "software",
        "status": "pending",
    },
    "EXP-004": {
        "employee": "Dave Wilson",
        "department": "Engineering",
        "description": "Keyboard and mouse for home office",
        "amount": 89.99,
        "category": "equipment",
        "status": "pending",
    },
}

# Company expense policy
EXPENSE_POLICY = {
    "auto_approve_limit": 500.00,  # Expenses under this amount are auto-approved
    "max_meal_expense": 150.00,
    "max_travel_expense": 2000.00,
    "max_software_expense": 1000.00,
    "max_equipment_expense": 500.00,
}

print("BigCompany Inc. expense database loaded!")
print(f"  - {len(EXPENSES_DB)} pending expenses")
print(f"  - Auto-approve limit: ${EXPENSE_POLICY['auto_approve_limit']}")
print()
for exp_id, exp in EXPENSES_DB.items():
    flag = "[AUTO]" if exp["amount"] <= EXPENSE_POLICY["auto_approve_limit"] else "[NEEDS HUMAN]"
    print(f"  {exp_id}: ${exp['amount']:>8.2f} - {exp['description'][:40]} {flag}")

BigCompany Inc. expense database loaded!
  - 4 pending expenses
  - Auto-approve limit: $500.0

  EXP-001: dollar 127.50 - Team lunch at downtown restaurant (AUTO)
  EXP-002: dollar 1250.00 - Flight to New York for client meeting (NEEDS HUMAN)
  EXP-003: dollar  659.88 - Annual Adobe Creative Cloud subscription (NEEDS HUMAN)
  EXP-004: dollar   89.99 - Keyboard and mouse for home office (AUTO)

## define the graph state

In [2]:
from typing_extensions import TypedDict

class ExpenseState(TypedDict): 
    #input 
    expense_id : str #which expense we're processing 


    # Filled in by the classification node 
    employee: str # whose submitted the expense
    description: str 
    amount: float 
    category: str # meals, travel, software, equipment 
    policy_compliant: bool 
    needs_human_review: bool 
    ai_recommendation: str

    decision:str
    decision_reason:str 


print("State defined: ExpenseState")
print(" Tracks: expense_id, employee, amount, category, policy_compliant,")
print("   needs_human_review, ai_recommendation, decision, decision_reason")

State defined: ExpenseState
 Tracks: expense_id, employee, amount, category, policy_compliant,
   needs_human_review, ai_recommendation, decision, decision_reason


## Set up the LLM

In [ ]:
from langchain_ollama import ChatOllama

# Initialize the LLM
llm = ChatOllama(
    model="llama3",   # or mistral, gemma, etc.
    temperature=0
)

print("LLM initialized: llama3 (Ollama)")

LLM initialized: gpt-4o-mini

## Graph Nodes

In [ ]:
from langgraph.types import interrupt
import json

# Node 1: Classify the Expense
def classify_expense(state: ExpenseState) -> dict:
    """Look up the expense and use AI to classify it."""
    expense_id = state["expense_id"]
    expense = EXPENSES_DB.get(expense_id)

    if not expense:
        return {
            "decision": "rejected",
            "decision_reason": f"Expense {expense_id} not found in database.",
        }

    # Check policy compliance using the LLM
    policy_prompt = f"""You are a finance assistant at BigCompany Inc.
Analyze this expense and determine if it complies with company policy.

Expense:
- Employee: {expense['employee']}   
- Description: {expense['description']}
- Amount: ${expense['amount']}
- Category: {expense['category']}

Company Policy Limits:
- Meals: max ${EXPENSE_POLICY['max_meal_expense']}
- Travel: max ${EXPENSE_POLICY['max_travel_expense']}
- Software: max ${EXPENSE_POLICY['max_software_expense']}
- Equipment: max ${EXPENSE_POLICY['max_equipment_expense']}

Respond in this exact JSON format (no markdown, just raw JSON):
{{
    "policy_compliant": true or false,
    "recommendation": "approve" or "reject",
    "reason": "brief explanation"
}}"""
    

    response = llm.invoke(policy_prompt)

    try:
        analysis = json.loads(response.content)
    except json.JSONDecodeError:
        analysis = {
            "policy_compliant": True,
            "recommendation": "approve",
            "reason": "Unable to parse AI analysis; defaulting to policy-compliant.",
        }

     # Determine if human review is needed (amount > $500)
    needs_human = expense["amount"] > EXPENSE_POLICY["auto_approve_limit"]

    print(f"\n[Classify] Expense {expense_id}:")
    print(f"  Employee: {expense['employee']}")
    print(f"  Amount: ${expense['amount']}")
    print(f"  Policy compliant: {analysis['policy_compliant']}")
    print(f"  AI recommendation: {analysis['recommendation']}")
    print(f"  Needs human review: {needs_human}")

    return {
        "employee": expense["employee"],
        "description": expense["description"],
        "amount": expense["amount"],
        "category": expense["category"],
        "policy_compliant": analysis["policy_compliant"],
        "needs_human_review": needs_human,
        "ai_recommendation": analysis["recommendation"],
    }


# Node 2: Human Review (with interrupt!)
def human_review(state: ExpenseState) -> dict:
    """Pause the workflow and ask a human manager to review the expense.

    This is where the magic happens!
    - interrupt() PAUSES the entire workflow.
    - It shows the human what needs to be reviewed.
    - The workflow waits (minutes, hours, or days) for the human to respond.
    - When the human responds via Command(resume=...), interrupt() RETURNS
      the human's answer, and the workflow continues.
    """

    # This is what the human will see when the workflow pauses
    review_request = {
        "message": "EXPENSE REVIEW REQUIRED",
        "expense_id": state["expense_id"],
        "employee": state["employee"],
        "description": state["description"],
        "amount": state["amount"],
        "category": state["category"],
        "policy_compliant": state["policy_compliant"],
        "ai_recommendation": state["ai_recommendation"],
        "instructions": "Reply with 'approved' or 'rejected: <reason>'",
    }

    print(f"\n[Human Review] Workflow PAUSED — waiting for human decision...")

    # *** THIS IS THE KEY LINE ***
    # interrupt() pauses execution and shows review_request to the human.
    # When the human responds, their answer becomes the return value.
    human_decision = interrupt(review_request)

    # Code below only runs AFTER the human responds via Command(resume=...)
    print(f"\n[Human Review] Human responded: {human_decision}")

    if human_decision.startswith("rejected"):
        reason = human_decision.replace("rejected:", "").strip()
        reason = reason if reason else "Rejected by manager."
        return {
            "decision": "rejected",
            "decision_reason": f"Manager decision: {reason}",
        }
    else:
        return {
            "decision": "approved",
            "decision_reason": "Approved by manager after review.",
        }


# Node 3: Process the Decision
def process_decision(state: ExpenseState) -> dict:
    """Record and display the final decision."""

    # In a real app, this would update the database and send notifications
    print(f"\n{'='*50}")
    print(f"EXPENSE DECISION REPORT")
    print(f"{'='*50}")
    print(f"  Expense ID:   {state['expense_id']}")
    print(f"  Employee:     {state['employee']}")
    print(f"  Description:  {state['description']}")
    print(f"  Amount:       ${state['amount']:.2f}")
    print(f"  Category:     {state['category']}")
    print(f"  Decision:     {state['decision'].upper()}")
    print(f"  Reason:       {state['decision_reason']}")
    print(f"{'='*50}")

    return {}


print("All nodes defined!")
print("  1. classify_expense — AI classifies and checks policy")
print("  2. human_review     — PAUSES for human approval (uses interrupt()!)")
print("  3. process_decision — Records the final decision")

All nodes defined!
  1. classify_expense — AI classifies and checks policy
  2. human_review     — PAUSES for human approval (uses interrupt()!)
  3. process_decision — Records the final decision

## Defining routing logic

In [ ]:
def route_expense(state: ExpenseState) -> str:
    """Decide whether the expense needs human review or can be auto-approved."""

    # If the expense was not found, go straight to process_decision
    if state.get("decision") == "rejected":
        return "process_decision"

    # If human review is needed (amount > $500), route to human_review node
    if state["needs_human_review"]:
        print(f"\n[Router] ${state['amount']:.2f} > $500 → routing to HUMAN REVIEW")
        return "human_review"

    # Otherwise, auto-approve
    print(f"\n[Router] ${state['amount']:.2f} ≤ $500 → AUTO-APPROVED")
    return "auto_approve"


def auto_approve(state: ExpenseState) -> dict:
    """Automatically approve small expenses that comply with policy."""
    return {
        "decision": "approved",
        "decision_reason": f"Auto-approved: amount ${state['amount']:.2f} is within auto-approve limit.",
    }


print("Routing logic defined!")
print("  ≤ $500 → auto_approve → process_decision")
print("  > $500 → human_review (PAUSES!) → process_decision")

Routing logic defined!
  more 500 → auto_approve → process_decision
  less 500 → human_review (PAUSES!) → process_decision

## Build the graph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# Create a new graph with our ExpenseState
builder = StateGraph(ExpenseState)

# --- Add Nodes ---
builder.add_node("classify_expense", classify_expense)
builder.add_node("human_review", human_review)
builder.add_node("auto_approve", auto_approve)
builder.add_node("process_decision", process_decision)

# --- Add Edges ---

# 1. Start → Classify: every expense begins with classification
builder.add_edge(START, "classify_expense")

# 2. Classify → (conditional): route based on amount
#    - If amount > $500 → human_review
#    - If amount ≤ $500 → auto_approve
#    - If expense not found → process_decision
builder.add_conditional_edges(
    "classify_expense",
    route_expense,
    {
        "human_review": "human_review",
        "auto_approve": "auto_approve",
        "process_decision": "process_decision",
    },
)

# 3. Both human_review and auto_approve lead to process_decision
builder.add_edge("human_review", "process_decision")
builder.add_edge("auto_approve", "process_decision")

# 4. process_decision → END
builder.add_edge("process_decision", END)

# --- Compile with Checkpointer ---
# InMemorySaver is REQUIRED for interrupt() to work!
# It saves the workflow state so it can be resumed later.
# (In production, use PostgresSaver for persistence across restarts)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

print("Graph compiled successfully!")
print("\nWorkflow:")
print("  START → classify_expense → [route] → auto_approve → process_decision → END")
print("                                     → human_review → process_decision → END")

Graph compiled successfully!

Workflow:
  START → classify_expense → [route] → auto_approve → process_decision → END
                                     → human_review → process_decision → END

## Test our Expense Approval Agent!

In [ ]:
# Process Alice's small expense
config_1 = {"configurable": {"thread_id": "expense-thread-1"}}

result = graph.invoke(
    {"expense_id": "EXP-001"},
    config_1,
)
 

(Classify) Expense EXP-001:
  Employee: Alice Johnson
  Amount: $127.5
  Policy compliant: True
  AI recommendation: approve
  Needs human review: False

(Router) dollar 127.50 ≤ $500 → AUTO-APPROVED

EXPENSE DECISION REPORT
  Expense ID:   EXP-001
  Employee:     Alice Johnson
  Description:  Team lunch at downtown restaurant
  Amount:       dollar 127.50
  Gategory:     meals
  Decision:     APPROVED
  Reason:       Auto-approved: amount ($127.50) is within auto-approve limit.

## Large expense human approve

In [ ]:
# Step A: Start processing Bob's expense — this will PAUSE
config_2 = {"configurable": {"thread_id": "expense-thread-2"}}

result = graph.invoke(
    {"expense_id": "EXP-002"},
    config_2,
)

print("\n>>> The workflow has PAUSED! <<<")
print(">>> It is waiting for a human manager to respond. <<<")


[Classify] Expense EXP-002:
  Employee: Bob Smith
  Amount: $1250.0
  Policy compliant: True
  AI recommendation: approve
  Needs human review: True

[Router] $1250.00 > $500 → routing to HUMAN REVIEW

[Human Review] Workflow PAUSED — waiting for human decision...

>>> The workflow has PAUSED! <<<
>>> It is waiting for a human manager to respond. <<<

In [ ]:
# Let's inspect the paused workflow state
paused_state = graph.get_state(config_2)

print("Workflow status:", paused_state.next)
print()

# The interrupt value is what was passed to interrupt()
# This is the information the human needs to make a decision
for task in paused_state.tasks:
    if hasattr(task, "interrupts") and task.interrupts:
        print("Information presented to the human manager:")
        print()
        review_info = task.interrupts[0].value
        for key, value in review_info.items():
            print(f"  {key}: {value}")

Workflow status: ('human_review',)

Information presented to the human manager:

  message: EXPENSE REVIEW REQUIRED
  expense_id: EXP-002
  employee: Bob Smith
  description: Flight to New York for client meeting
  amount: 1250.0
  category: travel
  policy_compliant: True
  ai_recommendation: approve
  instructions: Reply with 'approved' or 'rejected: <reason>'

## Concept of node

In [ ]:
def classify_expense(state: ExpenseState) ->dict:
    """look up the expense and use AI to classify it"""
    expense_id = state["expense_id"]
    expense = EXPENSES_DB.get(expense_id)

    if not expense: 
        return {
            "decision_reason" : f"Expense {expense_id} not found in database", 
            "decision":"rejected", 
        }
        
        # Check policy compliance using the LLM
    policy_prompt = f"""you are a finance assistant at BigCompany Inc.
Analyze this expense and determine if it complies with company policy.

Expense:
- Employee: {expense['employee']}
- Description: {expense['description']}
- Amount: ${expense['amount']}
- Category: {expense['category']}

Company Policy Limits:
- Meals: max ${EXPENSE_POLICY['max_meal_expense']}
- Travels: max ${EXPENSE_POLICY['max_travel_expense']}
- software : max ${EXPENSE_POLICY['max_software_expense']}
- Equipment : max ${EXPENSE_POLICY['max_equipment_expense']} 

Respond in this exact JSON format (no markdown, just raw JSON): 
{{
    "policy_compliant": true or false,
    "recommendation": "approve" or "reject",
    "reason": "brief explanation"
}}"""
    

    response = llm.invoke(policy_prompt)

    try:
        analysis = json.loads(response.content)
    except json.JSONDecodeError:
        analysis = {
            "policy_compliant": True,
            "recommendation": "approve",
            "reason": "Unable to parse AI analysis; defaulting to policy-compliant.",
        }
    
    # Determine if human review is needed (amount > $500)
    needs_human = expense["amount"] > EXPENSE_POLICY["auto_approve_limit"]

    print(f"\n[Classify] Expense {expense_id}:")
    print(f"  Employee: {expense['employee']}")
    print(f"  Amount: ${expense['amount']}")
    print(f"  Policy compliant: {analysis['policy_compliant']}")
    print(f"  AI recommendation: {analysis['recommendation']}")
    print(f"  Needs human review: {needs_human}")

    return {
        "employee": expense["employee"],
        "description": expense["description"],
        "amount": expense["amount"],
        "category": expense["category"],
        "policy_compliant": analysis["policy_compliant"],
        "needs_human_review": needs_human,
        "ai_recommendation": analysis["recommendation"],
    }
print("hello World")

In [ ]:
# for help

from langgraph.types import interrupt
import json


# ============================================================
# Node 1: Classify the Expense
# ============================================================
def classify_expense(state: ExpenseState) -> dict:
    """Look up the expense and use AI to classify it."""
    expense_id = state["expense_id"]
    expense = EXPENSES_DB.get(expense_id)

    if not expense:
        return {
            "decision": "rejected",
            "decision_reason": f"Expense {expense_id} not found in database.",
        }

    # Check policy compliance using the LLM
    policy_prompt = f"""You are a finance assistant at BigCompany Inc.
Analyze this expense and determine if it complies with company policy.

Expense:
- Employee: {expense['employee']}   
- Description: {expense['description']}
- Amount: ${expense['amount']}
- Category: {expense['category']}

Company Policy Limits:
- Meals: max ${EXPENSE_POLICY['max_meal_expense']}
- Travel: max ${EXPENSE_POLICY['max_travel_expense']}
- Software: max ${EXPENSE_POLICY['max_software_expense']}
- Equipment: max ${EXPENSE_POLICY['max_equipment_expense']}

Respond in this exact JSON format (no markdown, just raw JSON):
{{
    "policy_compliant": true or false,
    "recommendation": "approve" or "reject",
    "reason": "brief explanation"
}}"""
    

    response = llm.invoke(policy_prompt)

    try:
        analysis = json.loads(response.content)
    except json.JSONDecodeError:
        analysis = {
            "policy_compliant": True,
            "recommendation": "approve",
            "reason": "Unable to parse AI analysis; defaulting to policy-compliant.",
        }

     # Determine if human review is needed (amount > $500)
    needs_human = expense["amount"] > EXPENSE_POLICY["auto_approve_limit"]

    print(f"\n[Classify] Expense {expense_id}:")
    print(f"  Employee: {expense['employee']}")
    print(f"  Amount: ${expense['amount']}")
    print(f"  Policy compliant: {analysis['policy_compliant']}")
    print(f"  AI recommendation: {analysis['recommendation']}")
    print(f"  Needs human review: {needs_human}")

    return {
        "employee": expense["employee"],
        "description": expense["description"],
        "amount": expense["amount"],
        "category": expense["category"],
        "policy_compliant": analysis["policy_compliant"],
        "needs_human_review": needs_human,
        "ai_recommendation": analysis["recommendation"],
    }


# ============================================================
# Node 2: Human Review (with interrupt!)
# ============================================================
def human_review(state: ExpenseState) -> dict:
    """Pause the workflow and ask a human manager to review the expense.

    This is where the magic happens!
    - interrupt() PAUSES the entire workflow.
    - It shows the human what needs to be reviewed.
    - The workflow waits (minutes, hours, or days) for the human to respond.
    - When the human responds via Command(resume=...), interrupt() RETURNS
      the human's answer, and the workflow continues.
    """

    # This is what the human will see when the workflow pauses
    review_request = {
        "message": "EXPENSE REVIEW REQUIRED",
        "expense_id": state["expense_id"],
        "employee": state["employee"],
        "description": state["description"],
        "amount": state["amount"],
        "category": state["category"],
        "policy_compliant": state["policy_compliant"],
        "ai_recommendation": state["ai_recommendation"],
        "instructions": "Reply with 'approved' or 'rejected: <reason>'",
    }

    print(f"\n[Human Review] Workflow PAUSED — waiting for human decision...")

    # *** THIS IS THE KEY LINE ***
    # interrupt() pauses execution and shows review_request to the human.
    # When the human responds, their answer becomes the return value.
    human_decision = interrupt(review_request)

    # Code below only runs AFTER the human responds via Command(resume=...)
    print(f"\n[Human Review] Human responded: {human_decision}")

    if human_decision.startswith("rejected"):
        reason = human_decision.replace("rejected:", "").strip()
        reason = reason if reason else "Rejected by manager."
        return {
            "decision": "rejected",
            "decision_reason": f"Manager decision: {reason}",
        }
    else:
        return {
            "decision_reason": "Approved by manager after review.",
            "decision": "approved",
        }


# ============================================================
# Node 3: Process the Decision
# ============================================================
def process_decision(state: ExpenseState) -> dict:
    """Record and display the final decision."""

    # In a real app, this would update the database and send notifications
    print(f"\n{'='*50}")
    print(f"EXPENSE DECISION REPORT")
    print(f"{'='*50}")
    print(f"  Expense ID:   {state['expense_id']}")
    print(f"  Employee:     {state['employee']}")
    print(f"  Description:  {state['description']}")
    print(f"  Amount:       ${state['amount']:.2f}")
    print(f"  Category:     {state['category']}")
    print(f"  Decision:     {state['decision'].upper()}")
    print(f"  Reason:       {state['decision_reason']}")
    print(f"{'='*50}")

    return {}
        

print("All nodes defined!")
print("  1. classify_expense — AI classifies and checks policy")
print("  2. human_review     — PAUSES for human approval (uses interrupt()!)")
print("  3. process_decision — Records the final decision")

## Routings

In [ ]:
def route_expense(state: ExpenseState) -> str:
    """Decide whether the expense needs human review or can be auto-approved."""

    #If the expense was not found, go state in process_decision 
    if state.get("decision") == "rejected": 
        return "process_decision"

    # If human review is needed (amount>$500), route to human_review node 
    if state["needs_human_review"]: 
        print(f"\n[Router] ${state['amount']:.2f)} > $500 -> routing to human review")
        return "human_review" 

    #otherwise, auto-approve 
    # def auto_approve(state):
    #     return state
    print(f"\n[Router] ${state['amount']:.2f} <= $500 -> Auto-Approved")
    return "auto_approve"

    def auto_approve(state: ExpenseState) -> dict: 
        """Automatically approve small expenses that comply with policy."""
        return {
            "decision": "approve", 
            "decision_reason": f"Auto-approved:amount (${state['amount']:.2f}) is within auto-approve limit.",
        }
print("Routing logic defined!")
print("  ≤ $500 → auto_approve → process_decision")
print("  > $500 → human_review (PAUSES!) → process_decision")
        

In [ ]:
    
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# Create a new graph with our ExpenseState
builder = StateGraph(ExpenseState)

# --- Add Nodes ---
builder.add_node("classify_expense", classify_expense)
builder.add_node("human_review", human_review)
builder.add_node("auto_approve", auto_approve)
builder.add_node("process_decision", process_decision)

# --- Add Edges ---

# 1. Start → Classify: every expense begins with classification
builder.add_edge(START, "classify_expense")

# 2. Classify → (conditional): route based on amount
#    - If amount > $500 → human_review
#    - If amount ≤ $500 → auto_approve
#    - If expense not found → process_decision
builder.add_conditional_edges(
    "classify_expense",
    route_expense,
    {
        "human_review": "human_review",
        "auto_approve": "auto_approve",
        "process_decision": "process_decision",
    },
)

# 3. Both human_review and auto_approve lead to process_decision
builder.add_edge("human_review", "process_decision")
builder.add_edge("auto_approve", "process_decision")

# 4. process_decision → END
builder.add_edge("process_decision", END)

# --- Compile with Checkpointer ---
# InMemorySaver is REQUIRED for interrupt() to work!
# It saves the workflow state so it can be resumed later.
# (In production, use PostgresSaver for persistence across restarts)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

print("Graph compiled successfully!")
print("\nWorkflow:")
print("  START → classify_expense → [route] → auto_approve → process_decision → END")
print(" → human_review → process_decision → END")

### TEST

In [ ]:

# Process Alice's small expense
config_1 = {"configurable": {"thread_id": "expense-thread-1"}}

result = graph.invoke(
    {"expense_id": "EXP-001"},
    config_1,
)

In [ ]:
# Step A: Start processing Bob's expense — this will PAUSE
config_2 = {"configurable": {"thread_id": "expense-thread-2"}}

result = graph.invoke(
    {"expense_id": "EXP-002"},
    config_2,
)

print("\n>>> The workflow has PAUSED! <<<")
print(">>> It is waiting for a human manager to respond. <<<")

In [ ]:
# Step A: Start processing Bob's expense — this will PAUSE
config_2 = {"configurable": {"thread_id": "expense-thread-2"}}

result = graph.invoke(
    {"expense_id": "EXP-002"},
    config_2,
)

print("\n>>> The workflow has PAUSED! <<<")
print(">>> It is waiting for a human manager to respond. <<<")

## Large expense human reject

In [ ]:
# Inspect the final state of Bob's expense (approved)
final_state = graph.get_state(config_2)

print("Final state of expense-thread-2 (Bob's flight):")
print()
for key, value in final_state.values.items():
    print(f"  {key}: {value}")

print(f"\nWorkflow completed: {final_state.next == ()}")


(Classify) Expense EXP-003:
  Employee: Carol Davis
  Amount: $659.88
  Policy compliant: True
  AI recommendation: approve
  Needs human review: True

Router dollar 659.88 > dollar 500 → routing to HUMAN REVIEW

Human Review Workflow PAUSED — waiting for human decision...

 Workflow PAUSED — waiting for human decision...

## Understand the what happen under the hood

In [ ]:
# Inspect the final state of Bob's expense (approved)
final_state = graph.get_state(config_2)

print("Final state of expense-thread-2 (Bob's flight):")
print()
for key, value in final_state.values.items():
    print(f"  {key}: {value}")

print(f"\nWorkflow completed: {final_state.next == ()}")